# 04 Shoreline Quality Assurance

This notebook checks the corrected 2024 Mombasa shoreline candidates and supports visual review in QGIS. It does not create transects or calculate shoreline change. Automated checks alone do not confirm shoreline accuracy.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

cwd = Path.cwd()
if (cwd / "config").exists() and (cwd / "notebooks").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "config").exists() and (cwd.parent / "notebooks").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not resolve PROJECT_ROOT")

STUDY_AREA_PATH = PROJECT_ROOT / "config" / "study_area.geojson"
SCENE_METADATA_PATH = (
    PROJECT_ROOT / "data" / "interim" / "mombasa_test" / "selected_scene_metadata.csv"
)
WATER_MASK_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "mombasa_test"
    / "water_mask"
    / "water_mask_2024.tif"
)
MNDWI_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "mombasa_test"
    / "water_index"
    / "mndwi_2024.tif"
)
SHORELINE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mombasa_test"
    / "2024"
    / "shoreline_2024_shared_boundary_test.gpkg"
)
QA_DIR = PROJECT_ROOT / "outputs" / "qa" / "mombasa_test" / "2024"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
SHORELINE_LAYER = "shoreline_candidates"

for d in (QA_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
for name, path in [
    ("STUDY_AREA_PATH", STUDY_AREA_PATH),
    ("SCENE_METADATA_PATH", SCENE_METADATA_PATH),
    ("WATER_MASK_PATH", WATER_MASK_PATH),
    ("MNDWI_PATH", MNDWI_PATH),
    ("SHORELINE_PATH", SHORELINE_PATH),
]:
    print(f"{name}: {path} (exists={path.exists()})")

## Verify required inputs and load corrected shoreline

Check that all required files exist, confirm the shoreline layer, and load the corrected candidates for QA without modifying them.

In [ ]:
missing = [
    str(p)
    for p in (
        STUDY_AREA_PATH,
        SCENE_METADATA_PATH,
        WATER_MASK_PATH,
        MNDWI_PATH,
        SHORELINE_PATH,
    )
    if not p.exists()
]
if missing:
    raise FileNotFoundError("Missing required input(s):\n" + "\n".join(missing))

try:
    layers_table = gpd.list_layers(SHORELINE_PATH)
    print(layers_table)
    available_layers = layers_table["name"].tolist()
except Exception:
    import fiona

    available_layers = fiona.listlayers(str(SHORELINE_PATH))
print("Available layers:", available_layers)
if SHORELINE_LAYER not in available_layers:
    raise ValueError(
        f"Layer '{SHORELINE_LAYER}' not found. Available: {available_layers}"
    )

shoreline_candidates = gpd.read_file(SHORELINE_PATH, layer=SHORELINE_LAYER)
study_area = gpd.read_file(STUDY_AREA_PATH)
scene_metadata = pd.read_csv(SCENE_METADATA_PATH)

if shoreline_candidates.crs is None:
    raise ValueError("Shoreline CRS is missing")
if study_area.crs is None:
    raise ValueError("Study area CRS is missing")

for label, path in [("WATER_MASK", WATER_MASK_PATH), ("MNDWI", MNDWI_PATH)]:
    with rasterio.open(path) as src:
        print(label, "CRS:", src.crs)
        print(label, "Width/Height:", src.width, src.height)
        print(label, "Resolution:", src.res)
        print(label, "Bounds:", src.bounds)
        print(label, "NoData:", src.nodata)

print("Shoreline feature count:", len(shoreline_candidates))
print("Shoreline CRS:", shoreline_candidates.crs)
print(
    "Geometry types:", shoreline_candidates.geometry.geom_type.value_counts().to_dict()
)
print(shoreline_candidates.drop(columns=["geometry"]).head())
print(scene_metadata)

reported_count = 15802
if len(shoreline_candidates) != reported_count:
    print(
        f"WARNING: loaded count {len(shoreline_candidates)} differs from reported {reported_count}"
    )

## Verify raster alignment and invalid pixels

Confirm that water mask and MNDWI grids align and that invalid pixels remain distinguishable.

In [ ]:
with rasterio.open(WATER_MASK_PATH) as wm_src, rasterio.open(MNDWI_PATH) as mndwi_src:
    water_values = wm_src.read(1, masked=True)
    mndwi_values = mndwi_src.read(1, masked=True)
    water_mask_arr = np.ma.getmaskarray(water_values)
    mndwi_mask_arr = np.ma.getmaskarray(mndwi_values)
    wm_crs, mndwi_crs = wm_src.crs, mndwi_src.crs
    wm_shape, mndwi_shape = (
        (wm_src.height, wm_src.width),
        (mndwi_src.height, mndwi_src.width),
    )
    wm_transform, mndwi_transform = wm_src.transform, mndwi_src.transform
    wm_bounds, mndwi_bounds = wm_src.bounds, mndwi_src.bounds

print("CRS match:", wm_crs == mndwi_crs, wm_crs, mndwi_crs)
print("Shape match:", wm_shape == mndwi_shape, wm_shape, mndwi_shape)
print("Transform match:", wm_transform == mndwi_transform)
print("Bounds match:", wm_bounds == mndwi_bounds)

if not (
    wm_crs == mndwi_crs
    and wm_shape == mndwi_shape
    and wm_transform == mndwi_transform
    and wm_bounds == mndwi_bounds
):
    print("Grid mismatch detected; stopping pixel comparison")
    raise SystemExit

water_finite = np.isfinite(water_values.filled(np.nan))
water_valid = (~water_mask_arr) & water_finite
unique_water = np.unique(water_values.filled(np.nan)[water_valid])
print("Unique valid water-mask values:", unique_water)
if not set(unique_water.tolist()).issubset({0, 1}):
    raise ValueError("Unexpected valid water-mask values")

water_pixels_mask = water_valid & (water_values.filled(np.nan) == 1)
land_pixels_mask = water_valid & (water_values.filled(np.nan) == 0)
invalid_pixels_mask = ~water_valid

total_pixels = water_values.size
water_count = int(water_pixels_mask.sum())
land_count = int(land_pixels_mask.sum())
invalid_count = int(invalid_pixels_mask.sum())
valid_total = water_count + land_count
print("Total:", total_pixels)
print("Water:", water_count)
print("Land:", land_count)
print("Invalid:", invalid_count)
print("Valid %:", valid_total / total_pixels * 100)
print("Water % of valid:", (water_count / valid_total * 100) if valid_total else 0)
print("Land % of valid:", (land_count / valid_total * 100) if valid_total else 0)
assert water_count + land_count + invalid_count == total_pixels

for name, actual, expected in [
    ("Water", water_count, 2952072),
    ("Land", land_count, 1870198),
    ("Invalid", invalid_count, 95016),
]:
    print(
        name,
        "MATCH"
        if actual == expected
        else f"MISMATCH actual={actual} expected={expected}",
    )

water_valid_mndwi_invalid = int(
    (
        water_valid
        & (~np.isfinite(mndwi_values.filled(np.nan)) | np.ma.getmaskarray(mndwi_values))
    ).sum()
)
invalid_water_valid_mndwi = int(
    (
        invalid_pixels_mask
        & np.isfinite(mndwi_values.filled(np.nan))
        & (~np.ma.getmaskarray(mndwi_values))
    ).sum()
)
print("Valid water-mask pixels with invalid MNDWI:", water_valid_mndwi_invalid)
print("Invalid water-mask pixels with valid MNDWI:", invalid_water_valid_mndwi)
print("Invalid pixels in water mask:", int(invalid_pixels_mask.sum()))
print(
    "Invalid pixels in MNDWI:",
    int(
        (
            ~np.isfinite(mndwi_values.filled(np.nan)) | np.ma.getmaskarray(mndwi_values)
        ).sum()
    ),
)

if invalid_count == 0:
    print("WARNING: no invalid pixels found, conflicts with previous 95,016 count")
else:
    print("Invalid representation: stored mask and/or nonfinite values")

## Inspect geometry quality and segment lengths

Assess whether the large feature count is driven by short line segments. No geometries are modified.

In [ ]:
if shoreline_candidates.crs is None:
    raise ValueError("CRS is missing")
units = (
    str(shoreline_candidates.crs.axis_info[0].unit_name)
    if hasattr(shoreline_candidates.crs, "axis_info")
    else ""
)
epsg = shoreline_candidates.crs.to_epsg()
print("EPSG:", epsg, "Units:", units)
if "metre" not in units.lower() and "meter" not in units.lower():
    raise ValueError("CRS horizontal units are not metres")

print("Total features:", len(shoreline_candidates))
print("Null geometries:", shoreline_candidates.geometry.isna().sum())
print("Empty geometries:", shoreline_candidates.geometry.is_empty.sum())
print(
    "Invalid nonempty:",
    (
        ~shoreline_candidates.geometry.is_valid
        & ~shoreline_candidates.geometry.is_empty
        & shoreline_candidates.geometry.notna()
    ).sum(),
)
print(
    "Geometry types:", shoreline_candidates.geometry.geom_type.value_counts().to_dict()
)

inspect = shoreline_candidates[
    shoreline_candidates.geometry.notna()
    & ~shoreline_candidates.geometry.is_empty
    & shoreline_candidates.geometry.is_valid
].copy()
excluded = len(shoreline_candidates) - len(inspect)
print("Excluded features:", excluded)
if (
    ~shoreline_candidates.geometry.is_valid
    & shoreline_candidates.geometry.notna()
    & ~shoreline_candidates.geometry.is_empty
).any():
    print("QA issue: invalid geometries present")

inspect["orig_id"] = inspect.index
exploded = inspect.explode(index_parts=False).reset_index(drop=True)
exploded["length_m"] = exploded.geometry.length

print("Line parts:", len(exploded))
print("Zero-length parts:", int((exploded["length_m"] == 0).sum()))
print("Total km:", round(float(exploded["length_m"].sum()) / 1000, 4))
print("Min:", float(exploded["length_m"].min()) if len(exploded) else 0)
print("25th:", float(exploded["length_m"].quantile(0.25)) if len(exploded) else 0)
print("Median:", float(exploded["length_m"].median()) if len(exploded) else 0)
print("75th:", float(exploded["length_m"].quantile(0.75)) if len(exploded) else 0)
print("Max:", float(exploded["length_m"].max()) if len(exploded) else 0)

bins = [0, 10, 20, 50, 100, 500, np.inf]
labels = ["0-10", "10-20", "20-50", "50-100", "100-500", ">500"]
exploded["bin"] = pd.cut(
    exploded["length_m"], bins=bins, labels=labels, include_lowest=True
)
table = (
    exploded.groupby("bin", observed=True)
    .agg(count=("length_m", "size"), total_km=("length_m", lambda s: s.sum() / 1000))
    .reset_index()
)
table["pct"] = table["count"] / len(exploded) * 100 if len(exploded) else 0
print(table.to_string(index=False))

dup_count = exploded.geometry.duplicated().sum()
print("Exact duplicate geometries (coordinate duplicate check):", int(dup_count))

measured_km = float(exploded["length_m"].sum()) / 1000
if abs(measured_km - 265.95) > 0.01:
    print(f"Discrepancy: measured {measured_km:.4f} km vs 265.95 km")
else:
    print("Length within 0.01 km tolerance")

print("Ten shortest:")
print(
    exploded.nsmallest(10, "length_m")[['orig_id', 'length_m']].to_string(index=False)
)
print("Ten longest:")
print(exploded.nlargest(10, "length_m")[['orig_id', 'length_m']].to_string(index=False))

## Measure shoreline containment within the study area

Check whether corrected shoreline candidates extend outside the configured study area.
Containment is measured using geometric difference operations on line length. Point-only
contacts with the study-area boundary have zero length and are treated as contained.
Candidates lying on the study-area boundary are contained.

A containment pass here is not overall shoreline QA approval. Source geometries are never
clipped, repaired, simplified, or removed, and nothing is overwritten or exported.

In [ ]:
import shapely

CONTAINMENT_TOLERANCE_M = 0.01

# 1. Confirm both GeoDataFrames have a defined CRS and are nonempty.
if shoreline_candidates.crs is None:
    raise ValueError("shoreline_candidates CRS is missing")
if study_area.crs is None:
    raise ValueError("study_area CRS is missing")
if len(shoreline_candidates) == 0:
    raise ValueError("shoreline_candidates is empty")
if len(study_area) == 0:
    raise ValueError("study_area is empty")

shoreline_crs = shoreline_candidates.crs

# 3. Confirm shoreline CRS is projected with metre units.
units = (
    str(shoreline_crs.axis_info[0].unit_name)
    if hasattr(shoreline_crs, "axis_info")
    else ""
)
if "metre" not in units.lower() and "meter" not in units.lower():
    raise ValueError("Shoreline CRS horizontal units are not metres")

# 2. Create separate working copies; do not modify source GeoDataFrames.
sa = study_area.copy()
sl = shoreline_candidates.copy()

# Reproject study-area working copy to shoreline CRS.
sa = sa.to_crs(shoreline_crs)

In [ ]:
# 4. Validate study-area geometries.
valid_study_types = {"Polygon", "MultiPolygon"}
bad_types = ~sa.geometry.geom_type.isin(valid_study_types)
null_geoms = sa.geometry.isna()
empty_geoms = sa.geometry.is_empty
invalid_geoms = (~sa.geometry.is_valid) & (~empty_geoms) & (~null_geoms)

study_problem_rows = sa[bad_types | null_geoms | empty_geoms | invalid_geoms]
if len(study_problem_rows) > 0:
    problems = []
    for idx, geom in study_problem_rows.geometry.items():
        reason = []
        if geom is None or (
            hasattr(geom, "is_empty") and geom is not None and geom.is_empty
        ):
            reason.append("empty/null geometry")
        elif bad_types.loc[idx]:
            reason.append(f"unsupported geometry type '{geom.geom_type}'")
        if invalid_geoms.loc[idx]:
            reason.append("invalid geometry")
        problems.append((idx, "; ".join(reason)))
    raise ValueError(
        "Study area contains invalid geometries. Do not repair automatically. "
        "Investigate these rows:\n"
        + "\n".join(f"  index {i}: {r}" for i, r in problems)
    )

# 5. Combine study-area polygons into one geometry via union.
try:
    study_union = sa.geometry.unary_union
except Exception:
    study_union = shapely.union_all(list(sa.geometry))

# 6. Check shoreline geometries before measurement.
bad_sl_types = ~sl.geometry.geom_type.isin({"LineString", "MultiLineString"})
sl_null = sl.geometry.isna()
sl_empty = sl.geometry.is_empty
sl_invalid = (~sl.geometry.is_valid) & (~sl_empty) & (~sl_null)
sl_problem_rows = sl[bad_sl_types | sl_null | sl_empty | sl_invalid]
if len(sl_problem_rows) > 0:
    problems = []
    for idx, geom in sl_problem_rows.geometry.items():
        reason = []
        if geom is None or (
            hasattr(geom, "is_empty") and geom is not None and geom.is_empty
        ):
            reason.append("empty/null geometry")
        elif bad_sl_types.loc[idx]:
            reason.append(f"unsupported geometry type '{geom.geom_type}'")
        if sl_invalid.loc[idx]:
            reason.append("invalid geometry")
        problems.append((idx, "; ".join(reason)))
    raise ValueError(
        "Shoreline candidates contain invalid geometries. Do not repair automatically. "
        "Investigate these rows:\n"
        + "\n".join(f"  index {i}: {r}" for i, r in problems)
    )

In [ ]:
# 7. CONTAINMENT_TOLERANCE_M = 0.01 m (1 cm).
# This tolerance accommodates numerical precision only; it is not positional
# uncertainty and does not speak to shoreline accuracy.

# 8. Calculate containment metrics using geometric difference.
total_length_m = float(sl.geometry.length.sum())

outside_before = sl.geometry.difference(study_union)
outside_before_length_m = float(outside_before.length.sum())
outside_before_count = int((outside_before.length > 0).sum())

study_buffered = study_union.buffer(CONTAINMENT_TOLERANCE_M)
outside_after = sl.geometry.difference(study_buffered)
outside_after_length_m = float(outside_after.length.sum())
outside_after_count = int((outside_after.length > 0).sum())

In [ ]:
# 9. Print clearly labeled summary.
print("Shoreline CRS:", shoreline_crs)
print(f"Containment tolerance: {CONTAINMENT_TOLERANCE_M} m")
print(f"Total candidate length: {total_length_m / 1000:.4f} km")
print(f"Outside length before tolerance: {outside_before_length_m:.4f} m")
print(f"Outside length after tolerance: {outside_after_length_m:.4f} m")
print(f"Features outside before tolerance: {outside_before_count}")
print(f"Features outside after tolerance: {outside_after_count}")

# 10. Store summary in a one-row pandas DataFrame.
containment_summary = pd.DataFrame(
    [
        {
            "shoreline_crs": str(shoreline_crs),
            "containment_tolerance_m": CONTAINMENT_TOLERANCE_M,
            "total_candidate_length_km": total_length_m / 1000,
            "outside_length_before_tolerance_m": outside_before_length_m,
            "outside_length_after_tolerance_m": outside_after_length_m,
            "features_outside_before_tolerance": outside_before_count,
            "features_outside_after_tolerance": outside_after_count,
        }
    ]
)

# 11. Report affected features if outside linework remains after tolerance.
if outside_after_count > 0:
    sl_work = sl.copy()
    sl_work["outside_length_m"] = outside_after.length
    affected = sl_work[sl_work["outside_length_m"] > 0].copy()
    if "shoreline_id" in affected.columns:
        affected["feature_id"] = affected["shoreline_id"]
    else:
        affected["feature_id"] = affected.index
    affected = (
        affected[["feature_id", "outside_length_m"]]
        .sort_values("outside_length_m", ascending=False)
        .head(10)
    )
    print(
        "Containment issue requiring investigation: outside linework remains after tolerance"
    )
    print(affected.to_string(index=False))
else:
    print("Containment check passed within the stated tolerance.")

## Visual overview: corrected shoreline candidates over the water mask

Display corrected shoreline candidates over the water mask for visual QA. The map
shows candidate linework against land/water/invalid pixels and the study-area
boundary so the user can spot lines following NoData edges, the raster boundary,
or features outside the intended coastal scope.

This is a visual check only. It does not prove positional accuracy, and no
candidates are merged, simplified, repaired, filtered, or edited.

In [ ]:
import rasterio
from rasterio.plot import show
from rasterio.transform import from_bounds
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np

# 1. Open the water mask and read band 1 as masked array.
with rasterio.open(WATER_MASK_PATH) as src:
    raster_crs = src.crs
    transform = src.transform
    width = src.width
    height = src.height
    bounds = src.bounds
    water_band = src.read(1, masked=True)

water_data = np.ma.getdata(water_band).astype(float)
water_mask = np.ma.getmaskarray(water_band)
nonfinite = ~np.isfinite(water_data)
invalid = water_mask | nonfinite

# 3. Check north-up, unrotated transform.
a = transform.a
b = transform.b
c = transform.c
d = transform.d
e = transform.e
f = transform.f
if abs(b) > 1e-12 or abs(d) > 1e-12:
    raise ValueError(
        "Raster transform is rotated (b={} d={}). "
        "Cannot display at correct spatial coordinates without resampling. "
        "Stopping to avoid showing an incorrectly positioned image.".format(b, d)
    )
if a <= 0 or e >= 0:
    raise ValueError(
        "Raster transform is not north-up (a={} e={}). "
        "Stopping to avoid showing an incorrectly positioned image.".format(a, e)
    )

# 2. Create display-only copies in the raster CRS.
sa_disp = study_area.to_crs(raster_crs)
sl_disp = shoreline_candidates.to_crs(raster_crs)

# 5. Discrete two-class colormap: 0=land tan, 1=water light blue, invalid=white.
cmap = ListedColormap(["#D2B48C", "#ADD8E6", "#FFFFFF"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)

display = np.full(water_data.shape, 2.0, dtype=float)  # default invalid=2
display[~invalid] = water_data[~invalid]

# 4. Figure ~12 by 10 inches.
fig, ax = plt.subplots(figsize=(12, 10))

# Display water mask at correct spatial coordinates.
show(
    display,
    transform=transform,
    ax=ax,
    cmap=cmap,
    norm=norm,
    interpolation="nearest",
)

# 6. Overlay candidates and study-area boundary.
sl_disp.plot(ax=ax, color="red", linewidth=0.6, label="Candidate shoreline")
sa_disp.boundary.plot(
    ax=ax, color="dimgray", linestyle="--", linewidth=1.0, label="Study area"
)

# 7. Extent equal to raster bounds, equal axis scaling.
ax.set_xlim(bounds.left, bounds.right)
ax.set_ylim(bounds.bottom, bounds.top)
ax.set_aspect("equal", adjustable="box")

# 8. Title, date, subtitle, labels, CRS annotation.
acq_date = (
    scene_metadata["datetime"].iloc[0]
    if "datetime" in scene_metadata.columns
    else "unknown"
)
ax.set_title(
    f"Shoreline QA — Mombasa Test Area\nAcquisition: {acq_date}", fontsize=12, pad=12
)
ax.text(
    0.5,
    0.02,
    "Candidate boundaries — pending visual review",
    transform=ax.transAxes,
    ha="center",
    fontsize=10,
    style="italic",
    color="gray",
)

units = (
    str(raster_crs.axis_info[0].unit_name) if hasattr(raster_crs, "axis_info") else ""
)
is_metres = "metre" in units.lower() or "meter" in units.lower()
if is_metres:
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
else:
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

crs_id = raster_crs.to_epsg() if raster_crs.to_epsg() else str(raster_crs)
ax.annotate(
    f"CRS: {crs_id}",
    xy=(0.99, 0.01),
    xycoords="axes fraction",
    ha="right",
    va="bottom",
    fontsize=8,
    color="gray",
    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.7),
)

# Legend.
legend_elements = [
    Patch(facecolor="#ADD8E6", edgecolor="none", label="Water (1)"),
    Patch(facecolor="#D2B48C", edgecolor="none", label="Land (0)"),
    Patch(facecolor="#FFFFFF", edgecolor="gray", label="Invalid / NoData"),
    Line2D([0], [0], color="red", lw=0.6, label="Candidate shoreline"),
    Line2D([0], [0], color="dimgray", lw=1.0, ls="--", label="Study area boundary"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=8, framealpha=0.8)

# 9. Save figure.
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
out_path = FIGURES_DIR / "mombasa_test_shoreline_qa_overview_2024.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
print("Saved figure to:", out_path)
plt.show()

## Manual shoreline review log

This CSV log records observations made during QGIS inspection of the corrected
shoreline candidates. It is populated manually as issues are found; it is not
generated automatically.

Suggested issue types:
- NoData-edge artifact
- raster-edge artifact
- apparent classification error
- missing shoreline segment
- coastal-scope question

An empty log does not mean the shoreline passed review — it means visual review
is still pending.

In [ ]:
import pandas as pd

REVIEW_COLUMNS = [
    "review_id",
    "location_name",
    "issue_type",
    "severity",
    "observation",
    "recommended_action",
    "review_status",
]

REVIEW_LOG_PATH = QA_DIR / "manual_shoreline_review_2024.csv"

if REVIEW_LOG_PATH.exists():
    review_log = pd.read_csv(REVIEW_LOG_PATH)
else:
    review_log = pd.DataFrame(columns=REVIEW_COLUMNS)
    review_log.to_csv(REVIEW_LOG_PATH, index=False)

print(review_log)
print("Review log path:", REVIEW_LOG_PATH)
if len(review_log) == 0:
    print("No manual observations recorded yet; visual review is pending.")

# Inspect corrected shoreline in QGIS

1. Open the existing shoremonitor_qa project in QGIS.

2. Drag this file from Windows File Explorer into QGIS:
   data/processed/mombasa_test/2024/shoreline_2024_shared_boundary_test.gpkg

3. If prompted, select the shoreline_candidates layer.

4. Turn OFF the old shoreline_2024_test layer.

5. Place shoreline_candidates above water_mask_2024.
   Leave both switched ON.
   Turn OFF mndwi_2024 and all study_area layers.

6. Style the corrected candidates as thin red lines.

7. Zoom into a coastal white NoData gap.
   Toggle the corrected line layer off and on.

8. Record whether the corrected line:
   - Incorrectly follows the white gap's edge.
   - Stops where valid classified pixels end.
   - Resumes where a valid water–land boundary becomes available.

9. Repeat at two other coastal gaps.

Do not edit geometries during this inspection.
Do not connect gaps automatically.
Record actual observations in the manual review CSV.
Leave QA status pending until the inspection is completed.

## Record QGIS inspection observation

This records a targeted visual check of suspected coastal NoData-edge artifacts.
It does not establish positional accuracy or validate every candidate.

In [ ]:
import pandas as pd

observation = {
    "review_id": "QA_NODATA_001",
    "location_name": "Inland and coastal white gaps inspected in QGIS",
    "issue_type": "Suspected coastal NoData-edge artifact",
    "severity": "needs_investigation",
    "observation": (
        "red shoreline candidates do not "
        "outline inland white gaps, but appear to follow the edges of "
        "coastal white spots."
    ),
    "recommended_action": (
        "Inspect a coastal example closely and verify "
        "the displayed layer and underlying raster values before changing "
        "extraction logic."
    ),
    "review_status": "unresolved",
}

review_log = pd.read_csv(REVIEW_LOG_PATH)

mask = review_log["review_id"] == "QA_NODATA_001"
if mask.any():
    for col, val in observation.items():
        review_log.loc[mask, col] = val
else:
    review_log = pd.concat(
        [review_log, pd.DataFrame([observation])],
        ignore_index=True,
    )

review_log.to_csv(REVIEW_LOG_PATH, index=False)

print(review_log[review_log["review_id"] == "QA_NODATA_001"])

## Coastal gap raster-value inspection (folium)

Identify exact MNDWI and water-mask values at a coastal white spot that the red
shoreline candidates appear to follow. This does not modify shoreline extraction
logic; it only supports visual QA.

In [ ]:
import folium
import rasterio
from rasterio.warp import transform_bounds
import geopandas as gpd

MNDWI_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mombasa_test"
    / "2024"
    / "mndwi_2024.tif"
)
WATER_MASK_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mombasa_test"
    / "2024"
    / "water_mask_2024.tif"
)
SHORELINE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mombasa_test"
    / "2024"
    / "shoreline_2024_shared_boundary_test.gpkg"
)

with rasterio.open(MNDWI_PATH) as src:
    mndwi_crs = src.crs
    mndwi_bounds = src.bounds
    print("MNDWI CRS:", mndwi_crs)
    print("MNDWI Bounds:", mndwi_bounds)
    print("MNDWI Resolution:", src.res)
    print("MNDWI NoData:", src.nodata)

with rasterio.open(WATER_MASK_PATH) as src:
    wm_crs = src.crs
    wm_bounds = src.bounds
    print("Water mask CRS:", wm_crs)
    print("Water mask Bounds:", wm_bounds)
    print("Water mask Resolution:", src.res)
    print("Water mask NoData:", src.nodata)

try:
    layers_table = gpd.list_layers(SHORELINE_PATH)
    available_layers = layers_table["name"].tolist()
except Exception:
    import fiona
    available_layers = fiona.listlayers(str(SHORELINE_PATH))
print("Available GPKG layers:", available_layers)

shoreline_candidates = gpd.read_file(SHORELINE_PATH, layer="shoreline_candidates")
print("Shoreline CRS:", shoreline_candidates.crs)
print("Shoreline feature count:", len(shoreline_candidates))

sl_wgs84 = shoreline_candidates.to_crs(4326)

mndwi_bounds_wgs84 = transform_bounds(mndwi_crs, "EPSG:4326", *mndwi_bounds)
center_lat = (mndwi_bounds_wgs84[1] + mndwi_bounds_wgs84[3]) / 2
center_lon = (mndwi_bounds_wgs84[0] + mndwi_bounds_wgs84[2]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=13, control_scale=True)

mndwi_tile_url = "data/processed/mombasa_test/2024/mndwi_2024.tif"
wm_tile_url = "data/processed/mombasa_test/2024/water_mask_2024.tif"

folium.raster_layers.ImageOverlay(
    name="MNDWI 2024",
    image=mndwi_tile_url,
    bounds=[[mndwi_bounds_wgs84[1], mndwi_bounds_wgs84[0]], [mndwi_bounds_wgs84[3], mndwi_bounds_wgs84[2]]],
    opacity=0.7,
).add_to(m)

folium.raster_layers.ImageOverlay(
    name="Water Mask 2024",
    image=wm_tile_url,
    bounds=[[mndwi_bounds_wgs84[1], mndwi_bounds_wgs84[0]], [mndwi_bounds_wgs84[3], mndwi_bounds_wgs84[2]]],
    opacity=0.7,
).add_to(m)

folium.GeoJson(
    sl_wgs84,
    name="Shoreline Candidates",
    style_function=lambda x: {"color": "red", "weight": 1, "fillOpacity": 0},
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

QA_DIR = PROJECT_ROOT / "outputs" / "qa" / "2024"
QA_DIR.mkdir(parents=True, exist_ok=True)
map_path = QA_DIR / "shoreline_qa_coastal_gap_map.html"
m.save(str(map_path))

print("Saved interactive map to:", map_path)

print("""
QGIS INSPECTION INSTRUCTIONS:
1. Open the saved HTML map in a browser to locate a coastal white spot.
2. In QGIS, load the same three layers:
   - data/processed/mombasa_test/2024/mndwi_2024.tif
   - data/processed/mombasa_test/2024/water_mask_2024.tif
   - data/processed/mombasa_test/2024/shoreline_2024_shared_boundary_test.gpkg (shoreline_candidates)
3. Choose one coastal white spot where the red line follows the gap.
4. Use the Identify tool on the MNDWI layer.
5. Record MNDWI values around the white spot.
6. Use the Identify tool on the water-mask layer.
7. Record the water-mask values around the white spot.
8. Note whether the white spot is inside the study area.
9. Note whether the red line follows the white spot's outer edge
   or the edge of adjacent classified pixels.
""")